# OSS 4008 — LabelMe annotation conversion to YOLO format

**Purpose:** Convert manually verified hospital building boxes drawn with LabelMe into a training dataset.

**Status:** Implemented conversion pipeline; **requires actual image files and verified annotation JSON**. The notebook will not create fake labels.

Google Colab instructions: put images and matching LabelMe JSON files in `/content/hospital_raw/images/` and `/content/hospital_raw/labels/` before running the conversion cell. Use the Colab upload widget in the optional uploader cell or mount Drive.

**Important scientific limitation:** Hospitals cannot be distinguished from other institutional buildings by roof shape alone in many areas. Include background/negative patches and document ambiguity.

In [ ]:
%pip -q install Pillow pyyaml

In [ ]:
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split
import json, random, shutil, yaml
ROOT = Path('/content/hospital_raw')
IMAGES = ROOT / 'images'
LABELS = ROOT / 'labels'
OUTPUT = Path('/content/hospital_yolo')
for p in [IMAGES, LABELS, OUTPUT]: p.mkdir(parents=True, exist_ok=True)
print('Images folder:', IMAGES)
print('LabelMe JSON folder:', LABELS)

### Optional: upload a ZIP containing `images/` and `labels/`
Manual upload in Colab is required because Google Drive file paths may differ by account. ZIP extraction is restricted to this working directory.

In [ ]:
# Uncomment only if you have a ZIP of REAL imagery and matching LabelMe annotations.
# from google.colab import files
# import zipfile
# uploaded = files.upload()
# for name, content in uploaded.items():
#     if not name.lower().endswith('.zip'): continue
#     with zipfile.ZipFile(name) as z:
#         for member in z.infolist():
#             dest = (ROOT / member.filename).resolve()
#             if not dest.is_relative_to(ROOT.resolve()):
#                 raise ValueError('Unsafe ZIP member path')
#         z.extractall(ROOT)

In [ ]:
def convert_rectangle(points, width, height):
    if len(points) != 2:
        raise ValueError('LabelMe rectangle must contain two opposite corners')
    x1, y1 = [float(v) for v in points[0]]
    x2, y2 = [float(v) for v in points[1]]
    x1, x2 = sorted((max(0, min(width, x1)), max(0, min(width, x2))))
    y1, y2 = sorted((max(0, min(height, y1)), max(0, min(height, y2))))
    if x2 <= x1 or y2 <= y1:
        raise ValueError('Degenerate bounding box')
    return ((x1+x2)/(2*width), (y1+y2)/(2*height), (x2-x1)/width, (y2-y1)/height)

# A deterministic unit test on an illustrative box, NOT training imagery.
assert convert_rectangle([[10, 20], [30, 60]], 100, 100) == (0.2, 0.4, 0.2, 0.4)
print('Bounding-box conversion unit test passed')

In [ ]:
def convert_dataset():
    supported = {'.jpg', '.jpeg', '.png'}
    images = sorted(p for p in IMAGES.iterdir() if p.suffix.lower() in supported)
    if len(images) < 3:
        print('BLOCKED: Need at least 3 real annotated images to create train/val/test splits. Found:', len(images))
        return None
    examples = []
    warnings = []
    for img in images:
        annotation = LABELS / (img.stem + '.json')
        if not annotation.exists():
            warnings.append(f'Missing LabelMe JSON for {img.name}')
            continue
        data = json.loads(annotation.read_text(encoding='utf-8'))
        with Image.open(img) as photo:
            width, height = photo.size
        rows = []
        for shape in data.get('shapes', []):
            if shape.get('label', '').strip().lower() != 'hospital':
                warnings.append(f'Ignored nonhospital label in {annotation.name}')
                continue
            if shape.get('shape_type') != 'rectangle':
                warnings.append(f'Ignored nonrectangle shape in {annotation.name}')
                continue
            try:
                vals = convert_rectangle(shape['points'], width, height)
                rows.append('0 ' + ' '.join(f'{v:.6f}' for v in vals))
            except (ValueError, KeyError) as exc:
                warnings.append(f'Invalid rectangle: {annotation.name}: {exc}')
        examples.append((img, rows))
    if len(examples) < 3:
        print('BLOCKED: Not enough image/LabelMe pairs. Found:', len(examples)); print('\n'.join(warnings[:20])); return None
    rng = random.Random(4008)
    rng.shuffle(examples)
    n = len(examples)
    n_test = max(1, round(n*0.15))
    n_val = max(1, round(n*0.15))
    n_train = n - n_val - n_test
    if n_train < 1:
        raise ValueError('Insufficient train records')
    splits = {'train': examples[:n_train], 'val': examples[n_train:n_train+n_val], 'test': examples[n_train+n_val:]}
    for split, items in splits.items():
        image_dest = OUTPUT / 'images' / split
        label_dest = OUTPUT / 'labels' / split
        image_dest.mkdir(parents=True, exist_ok=True); label_dest.mkdir(parents=True, exist_ok=True)
        for image, rows in items:
            shutil.copy2(image, image_dest / image.name)
            (label_dest / (image.stem + '.txt')).write_text('\n'.join(rows) + ('\n' if rows else ''), encoding='utf-8')
    config = {'path': str(OUTPUT), 'train':'images/train','val':'images/val','test':'images/test','names':{0:'hospital'}}
    (OUTPUT / 'data.yaml').write_text(yaml.safe_dump(config, sort_keys=False),encoding='utf-8')
    print('Created verified-conversion dataset:', {k: len(v) for k,v in splits.items()})
    print('Warnings:', warnings[:20])
    print('YOLO config:', OUTPUT / 'data.yaml')
    return OUTPUT

DATASET_PATH = convert_dataset()

**Interpretation:** Successful conversion is **not** proof that a model can recognize hospital buildings; it merely prepares hand-verified training labels. Geographic leakage can inflate scores: split by hospital/site, not by adjacent image tiles, for final evaluation. The illustrative split above randomizes images and must be upgraded to site-disjoint grouping before reporting test accuracy.